In [ ]:
project_path = "/home/jupyter"
import os
import sys
sys.path.append(project_path)
from google.cloud import bigquery, storage

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px

from fintrans_toolbox.src import bq_utils as bq

In [ ]:
client = bigquery.Client()

In [ ]:
# Summarise the data by country
UK_spending_by_country3a = '''SELECT time_period_value, destination_country, spend FROM `ons-fintrans-data-prod.fintrans_visa.spend_origin_and_channel` where time_period = 'Quarter' and mcg = 'All' and merchant_channel = 'All' and cardholder_origin_country = 'All' and cardholder_origin = 'UNITED KINGDOM' and destination_country = 'UNITED KINGDOM' GROUP BY destination_country, 
time_period_value, spend ORDER BY time_period_value, destination_country DESC'''
df_by_country3a = bq.read_bq_table_sql(client, UK_spending_by_country3a)
df_by_country3a.head()

In [ ]:
import pandas as pd

# Assuming df_by_country3a is the DataFrame with your data
# Ensure 'time_period_value' is a string type and split it to get the year (assuming 'Q1', 'Q2', etc., are part of the time_period_value)

# Extract the year from the time_period_value (assuming it's in the format like '2023-Q1', '2023-Q2', etc.)
df_by_country3a['year'] = df_by_country3a['time_period_value'].str[:4].astype(int)

# Now group by year and sum the spend for each year
df_yearly_spend = df_by_country3a.groupby('year')['spend'].sum().reset_index()

# Optionally, you can sort the result by year
df_yearly_spend = df_yearly_spend.sort_values(by='year')

# Display the yearly totals
print(df_yearly_spend)

In [ ]:
df_by_country3a.to_csv('UK_spending_by_country3a.csv')

In [ ]:
# Display the yearly totals UK Spent in UK
df_yearly_spend.to_csv('UK_yearly_spend_country3a.csv')

In [ ]:
# Summarise the data by country
UK_spending_by_country3 = '''SELECT time_period_value, destination_country, spend FROM `ons-fintrans-data-prod.fintrans_visa.spend_origin_and_channel` where time_period = 'Quarter' and mcg = 'All' and merchant_channel = 'Online' and cardholder_origin_country = 'All' and cardholder_origin = 'UNITED KINGDOM' and destination_country = 'UNITED KINGDOM' GROUP BY destination_country, 
time_period_value, spend ORDER BY time_period_value, destination_country DESC'''
df_by_country3 = bq.read_bq_table_sql(client, UK_spending_by_country3)
# Rename the 'spend' column to 'online_spend'
df_by_country3 = df_by_country3.rename(columns={'spend': 'online_spend'})
df_by_country3.head()


In [ ]:
import pandas as pd

# Assuming df_by_country3a is the DataFrame with your data
# Ensure 'time_period_value' is a string type and split it to get the year (assuming 'Q1', 'Q2', etc., are part of the time_period_value)

# Extract the year from the time_period_value (assuming it's in the format like '2023-Q1', '2023-Q2', etc.)
df_by_country3['year'] = df_by_country3['time_period_value'].str[:4].astype(int)

# Now group by year and sum the spend for each year
df_yearly_spend1 = df_by_country3.groupby('year')['online_spend'].sum().reset_index()

# Optionally, you can sort the result by year
df_yearly_spend1 = df_yearly_spend1.sort_values(by='year')

# Display the yearly totals
print(df_yearly_spend1)

In [ ]:
df_yearly_spend1.to_csv('UK_yearly_spend_Online.csv')

In [ ]:
import pandas as pd

# Read the online spend CSV
df_online_spend = pd.read_csv('UK_yearly_spend_Online.csv')

# Read the total spend CSV
df_total_spend = pd.read_csv('UK_yearly_spend_country3a.csv')

# Display the first few rows of each DataFrame to check the structure
print(df_online_spend.head())
print(df_total_spend.head())

In [ ]:
# Merge the two DataFrames on 'year'
merged_spend = pd.merge(df_online_spend[['year', 'online_spend']], df_total_spend[['year', 'spend']], on='year', how='inner')

# Display the merged DataFrame to verify
print(merged_spend.head())


In [ ]:
# Calculate the online spend ratio (as a percentage)
merged_spend['online_spend_ratio'] = (merged_spend['online_spend'] / merged_spend['spend']) * 100

# Display the DataFrame with the new ratio column
print(merged_spend[['year', 'online_spend_ratio']])


In [ ]:
# Save the result to a new CSV file
merged_spend.to_csv('UK_yearly_online_spend_ratio.csv', index=False)

# Display a success message
print("The online spend ratio has been saved to 'UK_yearly_online_spend_ratio.csv'.")


In [ ]:
project_path = "/home/jupyter"
import os
import sys
sys.path.append(project_path)

from google.cloud import bigquery
import importlib
import plotly.express as px

import numpy as np
import pandas as pd
from datetime import datetime

import ft_digital_trade.src.utils.read_data as read_utils
import ft_digital_trade.src.utils.clean_utils as clean_utils
import ft_digital_trade.src.utils.calculation_utils as calc_utils
import ft_digital_trade.src.utils.plot_utils as plot_utils

client = bigquery.Client()

visa_data = read_utils.read_visa(cardholder_origin = "uk", cardholders_location = "uk", spend_location = "uk")

visa = calc_utils.calculate_visa(visa_data)
visa = clean_utils.rename_columns(df = visa, suffix = '_spoc')

global_cards = read_utils.read_global_cards()
global_cards = clean_utils.clean_global(global_cards)
global_cards = calc_utils.calculate_global(global_cards, 'card')

global_spend = read_utils.read_global_spend()
global_spend = clean_utils.clean_global(global_spend)
global_spend = calc_utils.calculate_global(global_spend, 'spend')

global_df = global_cards.merge(global_spend, how = 'inner', on = 'year', suffixes = ('_cards', '_spend'))
global_df = clean_utils.rename_columns(df = global_df, suffix = '_global')

uk_finance = read_utils.read_uk_finance()
uk_finance = clean_utils.clean_uk_finance(uk_finance)
uk_finance = calc_utils.calculate_uk_finance(uk_finance)
uk_finance = uk_finance[['year', 'cardholders','total value of purchases',"total volume of purchases"]]
uk_finance = clean_utils.rename_columns(df = uk_finance , suffix = '_uk_finance')

boe = read_utils.read_boe()
boe = clean_utils.clean_boe(boe)
boe = calc_utils.calculate_boe(boe)
boe = clean_utils.rename_columns(df = boe , suffix = '_boe')


In [ ]:
link = read_utils.read_link()

merged = visa.merge(uk_finance, how = 'outer', on = 'year')
merged = merged.merge(boe, how = 'outer', on = 'year')
merged = merged.merge(global_df, how = 'outer', on = 'year')

cardholders = merged[['year','cardholders_spoc','cardholders_uk_finance','visa_total_cards_global','total_cards_global', 'visa_marketshare_cards_global']]
cardholders = cardholders.copy()
cardholders['uk_finance_marketshare'] = cardholders['cardholders_spoc'] / cardholders['cardholders_uk_finance'] *100
cardholders['global_marketshare'] = cardholders['cardholders_spoc'] / cardholders['total_cards_global'] *100
#melt df for charts
cardholders = pd.melt(cardholders, id_vars='year',var_name='Data source', value_name='value')
cardholders = calc_utils.calculate_index(df = cardholders)

spend = merged[['year','spend_spoc', 
        'total value of purchases_uk_finance',
       'Mastercard values_boe', 'Visa Europe values_boe',
       'Mastercard and Visa values_boe', 'Visa proportion_boe',
       'debit_spend_global', 'credit_spend_global', 'visa_total_spend_global',
       'total_spend_global', 'visa_marketshare_spend_global']]
spend = spend.copy()
# #replace 2024 spending with NA
spend['spend_spoc'] = np.where(spend['year']==2024, np.nan, spend['spend_spoc'])
spend['total value of purchases_uk_finance'] = np.where(spend['year']==2024, np.nan, spend['total value of purchases_uk_finance'])
#calculate marketshare
spend['uk_finance_marketshare'] = spend['spend_spoc'] / spend['total value of purchases_uk_finance'] *100
spend['global_marketshare'] = spend['spend_spoc'] / spend['total_spend_global'] *100
spend['boe_marketshare'] = spend['spend_spoc'] / spend['Mastercard and Visa values_boe'] *100
#copy used for getting 2019 marketshare
spend_copy = spend.copy()
#melt df for charts
spend = pd.melt(spend, id_vars='year',var_name='Data source', value_name='value')
spend = calc_utils.calculate_index(df = spend)

plot_utils.plot_total_cardholders(df = cardholders)


In [ ]:
# Display the first few rows of the cardholders table
print(cardholders.head())



In [ ]:
# Display the first few rows of the spend table
print(spend.head())


In [ ]:
print(spend.columns)


In [ ]:
# Pivot the melted 'spend' dataframe to wide format
spend_wide = spend.pivot(index='year', columns='Data source', values='value')

# Now, 'spend_wide' will have columns like 'spend_spoc', 'uk_finance_marketshare', etc.
print(spend_wide.head())


In [ ]:
# Assuming the correct columns are present after pivoting
spend_wide['estimated_total_spending_uk'] = (spend_wide['spend_spoc'] / spend_wide['uk_finance_marketshare']) * spend_wide['total value of purchases_uk_finance']

spend_wide['estimated_total_spending_uk_global'] = (spend_wide['spend_spoc'] / spend_wide['global_marketshare']) * spend_wide['total_spend_global']

spend_wide['estimated_total_spending_uk_boe'] = (spend_wide['spend_spoc'] / spend_wide['boe_marketshare']) * spend_wide['Mastercard and Visa values_boe']


In [ ]:
# Replace NaNs with 0 (or handle as necessary)
spend_wide = spend_wide.fillna(0)


In [ ]:
# Pivot the melted 'spend' dataframe to wide format
spend_wide = spend.pivot(index='year', columns='Data source', values='value')

# Check the first few rows of the pivoted dataframe to ensure correct structure
print(spend_wide.head())

# Calculate the estimated total spending
spend_wide['estimated_total_spending_uk'] = (spend_wide['spend_spoc'] / spend_wide['uk_finance_marketshare']) * spend_wide['total value of purchases_uk_finance']
spend_wide['estimated_total_spending_uk_global'] = (spend_wide['spend_spoc'] / spend_wide['global_marketshare']) * spend_wide['total_spend_global']
spend_wide['estimated_total_spending_uk_boe'] = (spend_wide['spend_spoc'] / spend_wide['boe_marketshare']) * spend_wide['Mastercard and Visa values_boe']

# Optional: Handle missing values
spend_wide = spend_wide.fillna(0)

# View the results
print(spend_wide[['estimated_total_spending_uk', 'estimated_total_spending_uk_global', 'estimated_total_spending_uk_boe']])


In [ ]:
# Inspect the columns to find the relevant data
print(merged.columns)


In [ ]:
# Calculate Total Spend by UK Cardholders using the formula - Below NOT RIGHT


In [ ]:
# Calculate Total Spend by UK Cardholders using the formula
merged['total_spend_uk_cards'] = (merged['visa_marketshare_spend_global'] / merged['spend_spoc']) * 100


In [ ]:
# Optionally, replace NaNs with 0 or a suitable value
merged = merged.fillna(0)


In [ ]:
# Ensure that the columns used for calculation are not zero or NaN
merged = merged[merged['spend_spoc'] != 0]
merged['total_spend_uk_cards'] = (merged['visa_marketshare_spend_global'] / merged['spend_spoc']) * 100


In [ ]:
# Print the resulting 'total_spend_uk_cards' for verification
print(merged[['year', 'total_spend_uk_cards']])


In [ ]:
# Save the results to a CSV file
merged[['year', 'total_spend_uk_cards']].to_csv('total_spend_uk_cards.csv', index=False)



In [ ]:
import pandas as pd
import plotly.express as px

# Load the data from the CSV file
df = pd.read_csv('total_spend_uk_cards.csv')

# Convert 'total_spend_uk_cards' to numeric (in case it's formatted as strings like '£123,456')
df['total_spend_uk_cards'] = df['total_spend_uk_cards'].replace({'£': '', ',': ''}, regex=True).astype(float)

# Create a Bar chart
fig_bar = px.bar(df, x='year', y='total_spend_uk_cards', 
                 title="Total Spend by UK Cardholders (Bar Chart)", 
                 labels={'total_spend_uk_cards': 'Total Spend (£)', 'year': 'Year'},
                 text='total_spend_uk_cards')  # Display values on top of bars

# Customize the layout
fig_bar.update_layout(
    xaxis_title="Year",
    yaxis_title="Total Spend (£)",
    template="plotly_dark",  # Optional: dark theme
    showlegend=False
)

# Show the bar chart
fig_bar.show()

# Create a Line chart
fig_line = px.line(df, x='year', y='total_spend_uk_cards', 
                   title="Total Spend by UK Cardholders (Line Chart)",
                   labels={'total_spend_uk_cards': 'Total Spend (£)', 'year': 'Year'},
                   markers=True)  # Add markers to the line

# Customize the layout
fig_line.update_layout(
    xaxis_title="Year",
    yaxis_title="Total Spend (£)",
    template="plotly_dark",  # Optional: dark theme
    showlegend=False
)

# Show the line chart
fig_line.show()


In [ ]:
   year       online_spend_ratio
0  2019           40.728857
1  2020           49.799012
2  2021           48.471550
3  2022           46.433003
4  2023           47.173528
5  2024           47.078255


In [ ]:
   year      total_spend_uk_cards
0  2019          1.624782e-08
1  2020          1.622978e-08
2  2021          1.458940e-08
3  2022          1.262550e-08
4  2023          1.160988e-08
5  2024          0.000000e+00

In [ ]:
import pandas as pd

# Load the datasets
total_spend_df = pd.read_csv('total_spend_uk_cards.csv')
online_spend_ratio_df = pd.read_csv('UK_yearly_online_spend_ratio.csv')

# Merge the data on the 'Year' column
merged_df = pd.merge(total_spend_df, online_spend_ratio_df, on='year')


In [ ]:
# Calculate the online spend
merged_df['Online_Spend'] = merged_df['total_spend_uk_cards'] * merged_df['online_spend_ratio']


In [ ]:
total_online_spend = merged_df['Online_Spend'].sum()
print(f'Total UK Online Spend from 2019 to 2024: {total_online_spend}')


In [ ]:
import pandas as pd

# Load the data
total_spend_df = pd.read_csv('total_spend_uk_cards.csv')
online_spend_ratio_df = pd.read_csv('UK_yearly_online_spend_ratio.csv')

# Merge the data on the 'Year' column
merged_df = pd.merge(total_spend_df, online_spend_ratio_df, on='year')

# Calculate online spend for each year
merged_df['Online_Spend'] = merged_df['total_spend_uk_cards'] * merged_df['online_spend_ratio']

# Select only 'Year' and 'Online_Spend' for the final output
yearly_online_spend = merged_df[['year', 'Online_Spend']]

# Print the result (or save to a CSV if needed)
print(yearly_online_spend)


In [ ]:
# Save the result to a new CSV file
yearly_online_spend.to_csv('yearly_online_spend.csv', index=False)


In [ ]:
pip install matplotlib seaborn


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load the data
total_spend_df = pd.read_csv('total_spend_uk_cards.csv')
online_spend_ratio_df = pd.read_csv('UK_yearly_online_spend_ratio.csv')

# Merge the data on the 'Year' column
merged_df = pd.merge(total_spend_df, online_spend_ratio_df, on='year')

# Calculate online spend for each year
merged_df['Online_Spend'] = merged_df['total_spend_uk_cards'] * merged_df['online_spend_ratio']

# Select only 'Year' and 'Online_Spend' for the final output
yearly_online_spend = merged_df[['year', 'Online_Spend']]

# Plotting the Bar Chart and Line Chart

# Set up the figure and axes for both plots
fig, ax = plt.subplots(2, 1, figsize=(10, 10))

# Bar chart: Show total online spend per year
sns.barplot(x='year', y='Online_Spend', data=yearly_online_spend, ax=ax[0], palette='Blues_d')
ax[0].set_title('UK Online Spending per Year (Bar Chart)')
ax[0].set_xlabel('year')
ax[0].set_ylabel('Online Spend (£)')

# Line chart: Show total online spend trend over time
sns.lineplot(x='year', y='Online_Spend', data=yearly_online_spend, ax=ax[1], marker='o', color='green')
ax[1].set_title('UK Online Spending Trend (Line Chart)')
ax[1].set_xlabel('year')
ax[1].set_ylabel('Online Spend (£)')

# Adjust layout for better spacing
plt.tight_layout()

# Show the plots
plt.show()
